<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap12/12_4_Decoding_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 12.4: Decoding strategies**

This practical investigates neural decoding from transformer models.  

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [ ]:
!pip install transformers

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed
import torch
import torch.nn.functional as F
import numpy as np

In [ ]:
# Load model and tokenizer
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Decoding from GPT2

This tutorial investigates how to use GPT2 (the forerunner of GPT3) to generate text.  There are a number of ways to do this that trade-off the realism of the text against the amount of variation.

At every stage, GPT2 takes an input string and returns a probability for each of the possible subsequent tokens.  We can choose what to do with these probability.  We could always *greedily choose* the most likely next token, or we could draw a *sample* randomly according to the probabilities.  There are also intermediate strategies such as *top-k sampling* and *nucleus sampling*, that have some controlled randomness.

We'll also investigate *beam search* -- the idea is that rather than greedily take the next best token at each stage, we maintain a set of hypotheses  (beams)as we add each subsequent token and return the most likely overall hypothesis.  This is not necessarily the same result we get from greedily choosing the next token.

First, let's investigate the token themselves.  The code below prints out the vocabulary size and shows 20 random tokens.  

In [ ]:
np.random.seed(1)
print("Number of tokens in dictionary = %d"%(tokenizer.vocab_size))
for i in range(20):
  index = np.random.randint(tokenizer.vocab_size)
  print("Token: %d "%(index)+tokenizer.decode(torch.tensor(index), skip_special_tokens=True))


# Sampling

Each time we run GPT2 it will take in a set of tokens, and return a probability over each of the possible next tokens.  The simplest thing we could do is to just draw a sample from this probability distribution each time.

In [ ]:
def sample_next_token(input_tokens, model, tokenizer):
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]
  
  # TODO Draw a random token according to the probabilities
  # Implementation: use np.random.choice
  next_token = np.random.choice(len(prob_over_tokens), 1, p=prob_over_tokens)

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]

  return output_tokens

In [ ]:
# Expected output:
# "The best thing about Bath is that they don't even change or shrink anymore."

set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = sample_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))



In [ ]:
# TODO Modify the code below by changing the number of tokens generated and the initial sentence

input_txt = "In the future, artificial intelligence will"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(20):
    input_tokens = sample_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# Greedy token selection

You probably (correctly) got the impression that the text from pure sampling of the probability model can be kind of random.  How about if we choose most likely token at each step?


In [ ]:
def get_best_next_token(input_tokens, model, tokenizer):
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # TODO -- find the token index with the maximum probability
  # Implementation: use np.argmax
  next_token = [np.argmax(prob_over_tokens)]

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  return output_tokens

In [ ]:
# Expected output:
# The best thing about Bath is that it's a place where you can go to
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_best_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

In [ ]:
# TODO Experiment with a longer generation
input_txt = "The secret to a long life is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(20):
    input_tokens = get_best_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# Top-K sampling

Top-K sampling is a compromise strategy that samples randomly from the top K most probable tokens.  

In [ ]:
def get_top_k_token(input_tokens, model, tokenizer, k=20):
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # Draw a sample from the top K most likely tokens.
  # TODO -- Sort probabilities
  sorted_probs = np.sort(prob_over_tokens)[::-1]

  # TODO -- Find the probability at the k'th position
  kth_prob_value = sorted_probs[k-1]

  # Set all probabilities below this value to zero
  prob_over_tokens[prob_over_tokens < kth_prob_value] = 0

  # Renormalize
  # TODO -- divide by sum
  prob_over_tokens = prob_over_tokens / np.sum(prob_over_tokens)

  # Draw random token
  next_token = np.random.choice(len(prob_over_tokens), 1, replace=False, p=prob_over_tokens)

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  return output_tokens

In [ ]:
# Expected output:
# The best thing about Bath is that you get to see all the beautiful faces of

set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_top_k_token(input_tokens, model, tokenizer, k=10)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

In [ ]:
# TODO Experiment with k=50
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_top_k_token(input_tokens, model, tokenizer, k=50)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# Nucleus sampling

Nucleus sampling orders tokens by probability and cuts off when the cumulative sum reaches a threshold.


In [ ]:
def get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh=0.25):
  # Run model
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # TODO -- sort the probabilities in decreasing order
  sorted_probs_decreasing = np.sort(prob_over_tokens)[::-1]
  
  # TODO -- compute the cumulative sum
  cum_sum_probs = np.cumsum(sorted_probs_decreasing)

  # Find index where cumulative sum is greater than the threshold
  thresh_index = np.argmax(cum_sum_probs > thresh)
  # TODO: Find the probability value at this threshold index to use for filtering
  thresh_prob = sorted_probs_decreasing[thresh_index]

  # Set any probabilities less than this to zero
  prob_over_tokens[prob_over_tokens < thresh_prob] = 0
  # Renormalize
  prob_over_tokens = prob_over_tokens / np.sum(prob_over_tokens)
  # Draw random token
  next_token = np.random.choice(len(prob_over_tokens), 1, replace=False, p=prob_over_tokens)

  # Append
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  return output_tokens

In [ ]:
# Expected output:
# The best thing about Bath is that it's not a city that has been around
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh = 0.2)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))


In [ ]:
# TODO -- experiment with larger threshold (e.g., 0.9)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh = 0.9)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# Beam search

Beam search maintains $K$ hypotheses. 

In [ ]:
def get_kth_most_likely_token(input_tokens, model, tokenizer, k):
  # Run model
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # TODO Sort probabilities decreasingly
  sorted_probs = np.sort(prob_over_tokens)[::-1]
  # TODO Find the k'th sorted probability (index starts at 0)
  kth_prob_value = sorted_probs[k]

  # Find position
  next_token = np.where(prob_over_tokens == kth_prob_value)[0][0]
  next_token = np.array([next_token])

  # Append
  output_tokens = input_tokens.copy()
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  output_tokens['log_prob'] = output_tokens['log_prob'] + np.log(prob_over_tokens[next_token])
  return output_tokens

In [ ]:
# Test K=1
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
input_tokens['log_prob'] = 0.0
for i in range(10):
    input_tokens = get_kth_most_likely_token(input_tokens, model, tokenizer, k=1)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))


In [ ]:
# Print out each beam
def print_beams(beams):
  for index,beam in enumerate(beams):
    print("Beam %d, LogProb %3.3f: "%(index,beam['log_prob'])+tokenizer.decode(beam["input_ids"][0], skip_special_tokens=True))
  print('---')

def do_beam_search(input_tokens_in, model, tokenizer, n_beam=5, beam_length=10):
  input_tokens_in['log_prob'] = 0.0
  beams = [None] * n_beam
  for c_k in range(n_beam):
    beams[c_k] = get_kth_most_likely_token(input_tokens_in, model, tokenizer, c_k)

  for c_pos in range(beam_length-1):
    beams_all = []
    for c_beam in range(n_beam):
      for c_k in range(n_beam):
        beams_all.append(get_kth_most_likely_token(beams[c_beam], model, tokenizer, c_k))
    
    beams_all.sort(key=lambda x: x['log_prob'], reverse=True)
    beams = beams_all[:n_beam]
    print_beams(beams)

  return beams[0]

In [ ]:
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
best_beam = do_beam_search(input_tokens, model, tokenizer)
print("Final result:", tokenizer.decode(best_beam["input_ids"][0], skip_special_tokens=True))